# WCG Phase-1 plots

Loads CSVs from `../results/`, produces the four headline figures used in `docs/results.md`.

Run cells top-to-bottom. Requires `pandas` + `matplotlib`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path('../results').resolve()
WARM_MS = 5_000

LIMITER_ORDER = ['none', 'centralized_pertenant', 'gradient2_local', 'wcg']
LIMITER_COLOR = {
    'none': '#888888',
    'centralized_pertenant': '#1f77b4',
    'gradient2_local': '#2ca02c',
    'wcg': '#d62728',
}

def load(scenario, limiter):
    p = RESULTS / f'scenario_{scenario}__{limiter}.csv'
    return pd.read_csv(p)


## Figure 1 — Noisy tenant: per-tenant accepted RPS

The headline fairness chart. tA_noisy offers 100 RPS against a 20 RPS budget; tB_quiet offers 20 RPS within its 20 RPS budget. A fair limiter clamps tA at ~20 RPS without starving tB.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, kind in zip(axes, LIMITER_ORDER):
    df = load('noisy_tenant', kind)
    rps = df.groupby(['time_ms', 'tenant'])['accepted'].sum().unstack(fill_value=0)
    rps.index = rps.index / 1000
    rps.plot(ax=ax, lw=1.5)
    ax.set_title(kind)
    ax.set_xlabel('time (s)')
    ax.axhline(20, color='black', ls=':', alpha=0.4, label='budget = 20 RPS')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('accepted RPS (fleet)')
fig.suptitle('Noisy tenant: per-tenant goodput')
fig.tight_layout()
fig.savefig(RESULTS / 'fig1_noisy_tenant_per_tenant_rps.png', dpi=120)
plt.show()


## Figure 2 — Fleet p99 latency by scenario

p99 over time for every (scenario, limiter). The load-aware curves (gradient2, wcg) stay bounded; the non-load-aware curves explode whenever queueing kicks in.

In [ ]:
scenarios = ['noisy_tenant', 'heterogeneous', 'shock']
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, scn in zip(axes, scenarios):
    for kind in LIMITER_ORDER:
        df = load(scn, kind)
        p99 = df.groupby('time_ms')['p99_ms'].max() / 1000
        p99.index = p99.index / 1000
        ax.plot(p99.index, p99.values, lw=1.5, color=LIMITER_COLOR[kind], label=kind)
    ax.set_title(scn)
    ax.set_xlabel('time (s)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('fleet p99 (s, log scale)')
for ax in axes:
    ax.set_yscale('log')
axes[0].legend(loc='upper right', fontsize=8)
fig.suptitle('Fleet p99 latency')
fig.tight_layout()
fig.savefig(RESULTS / 'fig2_p99_by_scenario.png', dpi=120)
plt.show()


## Figure 3 — Shock: local_limit trajectory per server (WCG vs Gradient2)

srv-c slows down 3× at t=20s. Watch its local limit collapse while srv-a and srv-b stay healthy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, kind in zip(axes, ['gradient2_local', 'wcg']):
    df = load('shock', kind)
    lim = df.groupby(['time_ms', 'server'])['local_limit'].max().unstack()
    lim.index = lim.index / 1000
    lim.plot(ax=ax, lw=1.5)
    ax.axvline(20, color='red', ls='--', alpha=0.4, label='shock starts')
    ax.set_title(kind)
    ax.set_xlabel('time (s)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('local limit (C_i)')
fig.suptitle('Shock: per-server adaptive limit')
fig.tight_layout()
fig.savefig(RESULTS / 'fig3_shock_local_limit.png', dpi=120)
plt.show()


## Figure 4 — Jain's fairness index across all scenarios × limiters

Bar chart. 1.0 = perfect fairness. The WCG bars are the ones to watch.

In [ ]:
def jain(values):
    vs = [v for v in values if v > 0]
    if not vs:
        return float('nan')
    s = sum(vs)
    sq = sum(v*v for v in vs)
    return (s*s) / (len(vs) * sq)

rows = []
for scn in scenarios:
    for kind in LIMITER_ORDER:
        df = load(scn, kind)
        df = df[df['time_ms'] >= WARM_MS]
        per_tenant = df.groupby('tenant')['accepted'].sum()
        rows.append({'scenario': scn, 'limiter': kind, 'jain': jain(per_tenant.values)})
jdf = pd.DataFrame(rows).pivot(index='scenario', columns='limiter', values='jain')[LIMITER_ORDER]
ax = jdf.plot(kind='bar', figsize=(10, 4), color=[LIMITER_COLOR[k] for k in LIMITER_ORDER])
ax.set_ylabel("Jain's fairness index (1.0 = perfect)")
ax.set_ylim(0.5, 1.05)
ax.axhline(1.0, color='black', ls=':', alpha=0.4)
ax.set_title("Fairness across scenarios")
ax.legend(loc='lower right', fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS / 'fig4_jains_index.png', dpi=120)
plt.show()
jdf
